# 😀 SmileARC — Best-Result Reproduction Notebook

Single notebook that reproduces the **best smile-arc classifier** end-to-end:

1. Download dataset from HuggingFace
2. Extract frozen foundation features (DINOv2 ViT-B/14 + SigLIP ViT-B-16)
3. Generate fold-safe reverse-class augmentation (K=2)
4. Train MLP probe with logit adjustment + sqrt-sampling (5-fold OOF evaluation)
5. Auto label-error fixing (cleanlab-style, conservative)
6. Train final 3-seed ensemble on all data and save checkpoint

**Result: ~0.71 macro-F1** (0.667 before label fixes; label-fix number is an upper bound —
measured against cleaned targets). See README.md for the full experiment journey.

Runtime: ~5 min on a modern GPU (~102 GB VRAM used: <8 GB). Also works on CPU (slow: ~1 h).


In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────────
import os, subprocess, sys

def pip(pkg):
    try:
        __import__(pkg.split("[")[0].replace("-", "_"))
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for p in ["torch", "torchvision", "open_clip_torch", "huggingface_hub",
          "scikit-learn", "numpy", "pillow", "matplotlib"]:
    pip(p)

import numpy as np, torch, json, time, copy
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, classification_report
from PIL import Image
import torchvision.transforms.v2 as T2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch:", torch.__version__)

In [ ]:
# ── 1. Dataset: download + extract + verify ─────────────────────────────────
DATA_ROOT = "/marimo/data/smileArc"          # change if running elsewhere
CLASSES   = ["consonant", "not available", "reverse- non consonant", "straight- non consonant"]
EXPECTED  = [483, 431, 41, 131]              # per CLASSES order

def dataset_ready():
    if not os.path.isdir(DATA_ROOT): return False
    from collections import Counter
    c = Counter()
    for root, _, files in os.walk(DATA_ROOT):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                c[os.path.basename(root)] += 1
    return [c.get(cl, 0) for cl in CLASSES] == EXPECTED

if not dataset_ready():
    from huggingface_hub import hf_hub_download
    import zipfile
    zp = hf_hub_download(repo_id="aliabbas6622/smile",
                         filename="smileArc-20260830T185529Z-1-001.zip",
                         repo_type="dataset")
    os.makedirs(os.path.dirname(DATA_ROOT), exist_ok=True)
    with zipfile.ZipFile(zp) as z: z.extractall(os.path.dirname(DATA_ROOT))
    # zip may nest a folder — locate the real class dirs
    for root, dirs, _ in os.walk(os.path.dirname(DATA_ROOT)):
        if all(cl in dirs for cl in CLASSES) and root != os.path.dirname(DATA_ROOT):
            DATA_ROOT = root; break

from collections import Counter
c = Counter()
paths_by_class = {cl: [] for cl in CLASSES}
for cl in CLASSES:
    d = os.path.join(DATA_ROOT, cl)
    for f in sorted(os.listdir(d)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            paths_by_class[cl].append(os.path.join(d, f))
    c[cl] = len(paths_by_class[cl])
print("class counts:", dict(c), "| total:", sum(c.values()))
assert [c[cl] for cl in CLASSES] == EXPECTED, "unexpected class counts — check dataset" 

In [ ]:
# ── 2. Feature extraction: DINOv2 ViT-B/14 + SigLIP ViT-B-16 ────────────────
# DINOv2: CLS + patch-mean (768-d), ImageNet norm
# SigLIP: pooled image embedding (768-d), 0.5/0.5 norm
DINO_TF = T2.Compose([T2.Resize((224, 224)), T2.ToImage(), T2.ToDtype(torch.float32, scale=True),
                      T2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
SIG_TF  = T2.Compose([T2.Resize((224, 224)), T2.ToImage(), T2.ToDtype(torch.float32, scale=True),
                      T2.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

print("loading DINOv2 ViT-B/14 ...")
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", pretrained=True).eval().to(DEVICE)
print("loading SigLIP ViT-B-16 (webli) ...")
import open_clip
sig, _, _ = open_clip.create_model_and_transforms("ViT-B-16-SigLIP", pretrained="webli")
sig.eval().to(DEVICE)

@torch.no_grad()
def extract(paths, batch=32):
    fd, fs = [], []
    for s in range(0, len(paths), batch):
        imgs = [Image.open(p).convert("RGB") for p in paths[s:s+batch]]
        bd = torch.stack([DINO_TF(im) for im in imgs]).to(DEVICE)
        bs = torch.stack([SIG_TF(im) for im in imgs]).to(DEVICE)
        od = dino(bd)
        if isinstance(od, dict):
            f = np.concatenate([od["x_norm_clstoken"].cpu().numpy(),
                                od["x_norm_patchtokens"].mean(1).cpu().numpy()], 1)
        else:
            f = od.cpu().numpy()
        fd.append(f); fs.append(sig.encode_image(bs).float().cpu().numpy())
    return np.concatenate(fd, 0).astype(np.float32), np.concatenate(fs, 0).astype(np.float32)

all_paths = [p for cl in CLASSES for p in paths_by_class[cl]]
y = np.array([i for i, cl in enumerate(CLASSES) for _ in paths_by_class[cl]])
t0 = time.time()
dino_f, sig_f = extract(all_paths)
X = np.concatenate([dino_f, sig_f], 1)          # (1086, 1536) base features
print(f"features: dino{dino_f.shape} sig{sig_f.shape} -> X{X.shape} in {time.time()-t0:.0f}s")

In [ ]:
# ── 3. Reverse-class augmentation (fold-safe) ───────────────────────────────
# 8 deterministic variants per reverse image; only ever added to the TRAIN side
# of each fold, so validation folds stay clean (no leakage).
REV_IDX  = [i for i, c in enumerate(CLASSES) if "reverse" in c.lower()][0]
rev_rows = np.where(y == REV_IDX)[0]

def augment_pil(img, k):
    import random
    rng = random.Random(1000 + k)
    w, h = img.size
    im = img
    im = T2.functional.adjust_brightness(im, 1.0 + rng.uniform(-0.25, 0.25))
    im = T2.functional.adjust_contrast(im, 1.0 + rng.uniform(-0.25, 0.25))
    im = T2.functional.adjust_saturation(im, 1.0 + rng.uniform(-0.15, 0.15))
    ang = [-12, 12, -6, 6, -9, 9, 3, -3][k % 8]
    im = im.rotate(ang, resample=Image.BILINEAR, expand=False)
    dx, dy = rng.uniform(-0.06, 0.06) * w, rng.uniform(-0.06, 0.06) * h
    im = im.transform((w, h), Image.AFFINE, (1, 0, dx, 0, 1, dy), resample=Image.BILINEAR)
    sx = 1.0 + rng.uniform(-0.08, 0.08); sy = 1.0 + rng.uniform(-0.08, 0.08)
    im = im.resize((max(8, int(w * sx)), max(8, int(h * sy)))).resize((w, h))
    if k % 2 == 1: im = im.transpose(Image.FLIP_LEFT_RIGHT)
    return im

N_AUG = 8
aug_imgs, aug_meta = [], []          # (row_id, k)
for rid in rev_rows:
    img = Image.open(all_paths[rid]).convert("RGB")
    for k in range(N_AUG):
        aug_imgs.append(augment_pil(img, k)); aug_meta.append((int(rid), k))
print(f"generated {len(aug_imgs)} augmented reverse images ({N_AUG} per original x {len(rev_rows)})")

t0 = time.time()
import tempfile
dino_a, sig_a = [], []
tmpdir = tempfile.mkdtemp()
for s in range(0, len(aug_imgs), 32):
    chunk = aug_imgs[s:s+32]
    ps = []
    for j, im in enumerate(chunk):
        p = os.path.join(tmpdir, f"a{s+j}.jpg"); im.save(p); ps.append(p)
    d, sg = extract(ps)
    dino_a.append(d); sig_a.append(sg)
aug_dino = np.concatenate(dino_a, 0).astype(np.float32)
aug_sig  = np.concatenate(sig_a, 0).astype(np.float32)
aug_X    = np.concatenate([aug_dino, aug_sig], 1)     # (328, 1536)
aug_rows = np.array([m[0] for m in aug_meta])
aug_ks   = np.array([m[1] for m in aug_meta])
print(f"aug features: {aug_X.shape} in {time.time()-t0:.0f}s")

In [ ]:
# ── 4. Probe training (locked v5 recipe) ────────────────────────────────────
HP = dict(epochs=200, lr=1e-3, wd=1e-3, patience=30, dropout=0.4, batch=128, n_folds=5)

def make_probe(in_dim, n_classes=4):
    return nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, 256), nn.GELU(),
                         nn.Dropout(HP["dropout"]), nn.Linear(256, n_classes))

def train_probe(Xtr_all, ytr_all, seed=0, epochs=None, patience=None):
    """Train one probe: scaler, sqrt-weighted sampler, class-weighted CE,
    logit adjustment by log-prior (tau=1.0), early stop on val macro-F1.
    Pass (Xva, yva) via val_data for early stopping; else train full epochs."""
    epochs = epochs or HP["epochs"]; patience = patience or HP["patience"]
    torch.manual_seed(seed * 1000 + 7)
    sc = StandardScaler().fit(Xtr_all)
    Xs = sc.transform(Xtr_all).astype(np.float32)
    yt = torch.tensor(ytr_all, dtype=torch.long)
    cc = np.maximum(np.bincount(ytr_all, minlength=4), 1)
    sw = (1.0 / np.sqrt(cc))[ytr_all]
    gen = torch.Generator(); gen.manual_seed(seed * 1000 + 7)
    sampler = WeightedRandomSampler(torch.tensor(sw, dtype=torch.double),
                                    num_samples=len(ytr_all), replacement=True, generator=gen)
    dl = DataLoader(TensorDataset(torch.tensor(Xs), yt), batch_size=HP["batch"], sampler=sampler)
    model = make_probe(Xs.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    cw = torch.tensor(cc.sum() / (4.0 * cc), dtype=torch.float32).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw)
    log_prior = np.log(np.bincount(y, minlength=4) / len(y) + 1e-8)
    bias = torch.tensor(1.0 * log_prior, dtype=torch.float32).to(DEVICE)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb) + bias, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        sched.step()
    model.eval()
    return model, sc, bias

@torch.no_grad()
def predict_probs(model, sc, bias, Xva):
    Xs = sc.transform(Xva).astype(np.float32)
    xb = torch.tensor(Xs, dtype=torch.float32).to(DEVICE)
    return F.softmax(model(xb) + bias, 1).cpu().numpy()

def gmean(y_true, y_pred):
    rec = recall_score(y_true, y_pred, average=None, zero_division=0)
    return float(np.exp(np.mean(np.log(rec + 1e-8))))

def evaluate(X_base, y_true, K=2, tag=""):
    """5-fold OOF with K augmented copies of train-side reverse rows."""
    oof = np.zeros((len(y_true), 4), dtype=np.float32)
    skf = StratifiedKFold(n_splits=HP["n_folds"], shuffle=True, random_state=42)
    row2aug = {}
    for i, rid in enumerate(aug_rows): row2aug.setdefault(int(rid), []).append(i)
    for fi, (tr, va) in enumerate(skf.split(X_base, y_true)):
        tr_set = set(tr.tolist())
        sel = []
        for rid, idxs in row2aug.items():
            if rid in tr_set: sel.extend(idxs[:K])
        sel = np.array(sorted(sel), dtype=int)
        Xtr = np.concatenate([X_base[tr], aug_X[sel]], 0)
        ytr = np.concatenate([y_true[tr], np.full(len(sel), REV_IDX, dtype=y_true.dtype)], 0)
        m, sc, b = train_probe(Xtr, ytr, seed=fi)
        oof[va] = predict_probs(m, sc, b, X_base[va])
    pred = oof.argmax(1)
    f1m = f1_score(y_true, pred, average="macro", zero_division=0)
    f1r = f1_score(y_true == REV_IDX, pred == REV_IDX, zero_division=0)
    print(f"{tag:24s} macro-F1={f1m:.4f}  reverse-F1={f1r:.4f}  G-mean={gmean(y_true, pred):.4f}")
    return oof, f1m, f1r

print("== A/B: augmentation sweep (identical folds) ==")
for K, tag in [(0, "baseline K=0"), (2, "K=2 aug (winner)"), (4, "K=4 aug")]:
    evaluate(X, y, K=K, tag=tag)
results_ab = evaluate(X, y, K=2, tag="K=2 (final OOF)")

In [ ]:
# ── 5. Label-error auto-fix (conservative, cleanlab-style) ──────────────────
# Rank suspects by OOF confidence margin; apply a fix ONLY when the ensemble is
# near-certain (margin >= 0.90) across folds, capped at 40 fixes, arc classes only.
oof_probs, _, _ = results_ab
margin = 1 - oof_probs[np.arange(len(y)), y]          # prob mass NOT on the gold label
order  = np.argsort(-margin)

ARC_PAIRS = {i for i, c in enumerate(CLASSES) if "consonant" in c.lower() or "straight" in c.lower()}
y_fixed = y.copy(); fixes = []
for i in order[:40]:
    pred = int(oof_probs[i].argmax())
    if pred == y[i] or margin[i] < 0.90: continue
    if not (y[i] in ARC_PAIRS and pred in ARC_PAIRS): continue
    if oof_probs[i, pred] < 0.90: continue
    fixes.append((all_paths[i], CLASSES[y[i]], CLASSES[pred], float(margin[i])))
    y_fixed[i] = pred

print(f"applied {len(fixes)} label fixes (margin>=0.90):")
for p, old, new, m in fixes: print(f"  {os.path.basename(p):40s} {old:28s} -> {new:28s} margin={m:.3f}")

print("\n== re-evaluate on cleaned labels ==")
oof_clean, f1_clean, f1r_clean = evaluate(X, y_fixed, K=2, tag="K=2 + cleaned labels")

> ⚠️ **Honest caveat:** the cleaned-labels score is measured against the cleaned targets
> themselves, so it is optimistic. The true apples-to-apples number is the baseline above;
> human review of the flagged labels (`fixes` list) is the way to confirm the gain.

In [ ]:
# ── 6. Final model: train on ALL data + K=2 aug, 3-seed ensemble ────────────
MODELS_DIR = os.environ.get("SMILEARC_MODELS_DIR", "./models")
os.makedirs(MODELS_DIR, exist_ok=True)

K_FINAL = 2
row2aug = {}
for i, rid in enumerate(aug_rows): row2aug.setdefault(int(rid), []).append(i)
sel_all = sorted([i for idxs in row2aug.values() for i in idxs[:K_FINAL]])
X_all = np.concatenate([X, aug_X[np.array(sel_all)]], 0)
y_all = np.concatenate([y_fixed, np.full(len(sel_all), REV_IDX, dtype=y_fixed.dtype)], 0)
print(f"final training set: {X_all.shape}")

final_models = []
for s in range(3):
    m, sc, b = train_probe(X_all, y_all, seed=100 + s)
    final_models.append((m, sc, b))
    print(f"  seed {s} trained")

model_path = os.path.join(MODELS_DIR, "smile_arc_v5_final_K2aug_3seed.pt")
torch.save({
    "state_dicts": [m.state_dict() for m, _, _ in final_models],
    "scalers":     [(sc.mean_, sc.scale_) for _, sc, _ in final_models],
    "biases":      [b.cpu().numpy() for _, _, b in final_models],
}, model_path)
print("saved ->", model_path)

info = {
    "name": "smile_arc_v5_final_K2aug_3seed",
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "classes": CLASSES,
    "features": "concat(DINOv2 ViT-B/14 cls+patchmean 768d, SigLIP ViT-B-16-SigLIP-webli 768d) = 1536d",
    "head": "LayerNorm -> Linear(1536,256) -> GELU -> Dropout(0.4) -> Linear(256,4)",
    "augmentation": "reverse-class offline aug K=2, train-side only",
    "sampling": "WeightedRandomSampler 1/sqrt(class_count); CE + inverse-freq class weights",
    "logit_adjustment": "logits + 1.0 * log(class_prior)",
    "optimizer": "AdamW lr=1e-3 wd=1e-3, cosine T=200",
    "ensemble": "3 seeds (100,101,102) soft-vote",
    "label_fixes": len(fixes),
    "metrics": {
        "oof_macro_f1_baseline": round(float(results_ab[1]), 4),
        "oof_macro_f1_cleaned": round(float(f1_clean), 4),
        "oof_reverse_f1_baseline": round(float(results_ab[2]), 4),
        "oof_reverse_f1_cleaned": round(float(f1r_clean), 4),
    },
}
with open(os.path.join(MODELS_DIR, "smile_arc_v5_final_K2aug_3seed_info.json"), "w") as fh:
    json.dump(info, fh, indent=2)
print(json.dumps(info["metrics"], indent=2))

In [ ]:
# ── 7. Inference helper: predict on new images with the saved ensemble ──────
@torch.no_grad()
def predict(images):
    """images: list of paths or PIL images. Returns (paths, class probs)."""
    if isinstance(images, (str, Image.Image)): images = [images]
    paths = [im if isinstance(im, str) else f"pil_{i}" for i, im in enumerate(images)]
    imgs = [Image.open(p).convert("RGB") if isinstance(p, str) else im for p, im in zip(paths, images)]
    d, s = extract_dino_sig(imgs)
    Xt = torch.tensor(np.concatenate([d, s], 1), dtype=torch.float32)
    prob = None
    for m, sc, b in final_models:
        xs = torch.tensor(sc.transform(Xt.numpy()).astype(np.float32)).to(DEVICE)
        p = F.softmax(m(xs) + b, 1).cpu().numpy()
        prob = p if prob is None else prob + p
    return paths, prob / len(final_models)

# reuse extract() internals for PIL images
def extract_dino_sig(imgs, batch=32):
    fd, fs = [], []
    for st in range(0, len(imgs), batch):
        chunk = imgs[st:st+batch]
        bd = torch.stack([DINO_TF(im) for im in chunk]).to(DEVICE)
        bs = torch.stack([SIG_TF(im) for im in chunk]).to(DEVICE)
        od = dino(bd)
        f = (np.concatenate([od["x_norm_clstoken"].cpu().numpy(),
                             od["x_norm_patchtokens"].mean(1).cpu().numpy()], 1)
             if isinstance(od, dict) else od.cpu().numpy())
        fd.append(f); fs.append(sig.encode_image(bs).float().cpu().numpy())
    return np.concatenate(fd, 0).astype(np.float32), np.concatenate(fs, 0).astype(np.float32)

# demo: one image per class
for cl in CLASSES:
    p = paths_by_class[cl][0]
    _, pr = predict(p)
    top = int(pr[0].argmax())
    print(f"{os.path.basename(p):30s} true={cl:26s} pred={CLASSES[top]:26s} p={pr[0][top]:.3f}")